# PAM-ROFI quickstart (Colab or Jupyter)
Runs the whole pipeline and shows the maps with geemap. The web app (`streamlit run app/streamlit_app.py`) does the same with more features.

In [ ]:
# Colab only: get the code
# !git clone https://github.com/manasmmm/PAM-ROFI.git && %cd PAM-ROFI
# !pip install -q -r requirements.txt geemap

In [ ]:
import sys; sys.path.insert(0, '..') if 'notebooks' in __import__('os').getcwd() else None
from pamrofi import earthengine, viz
from pamrofi.config import Config
from pamrofi.pipeline import Pipeline, PipelineOptions

cfg = Config.load()
earthengine.init(cfg.ee_project)
pipe = Pipeline(cfg, PipelineOptions(roi_key='meerut_buffer_10km'))
print(pipe.season.label, '| validated labels:', pipe.validated)

In [ ]:
# Accuracy (meaningful only with real labels)
m = pipe.evaluate()
print('OA', round(m['overall_accuracy']*100, 2), '% | kappa', round(m['kappa'], 3))
m['per_class']

In [ ]:
import geemap
DATE = '2025-09-10'
Map = geemap.Map()
Map.centerObject(pipe.roi, 12)
Map.addLayer(pipe.crop_map, viz.crop_vis(pipe.season), 'Crop type')
Map.addLayer(pipe.stress(DATE).select('stress'), viz.STRESS_VIS, 'Moisture stress', False)
Map.addLayer(pipe.advisory(DATE).select('advisory'), viz.ADVISORY_VIS, 'Irrigation advisory', False)
Map.add_legend(title='Crop type', legend_dict=viz.crop_legend(pipe.season))
Map

In [ ]:
import pandas as pd
pd.DataFrame(pipe.advisory_summary(DATE))

In [ ]:
ts = pipe.point_timeseries(77.685, 29.117)
pd.DataFrame(ts['composites'])[['date','NDVI','VH','PAM_stage','VCI','SMI','stress']]